# Task 05 — Time-Series Forecasting with ARIMA & Prophet

**End-to-End Demand Forecasting Pipeline: Trend, Seasonality, Stationarity, Model Training & 30-Day Projections**

This notebook implements an end-to-end time-series analysis and forecasting pipeline for daily product demand, evaluating classical Box-Jenkins (SARIMAX) and additive Bayesian (Facebook Prophet) architectures.

## 1. Objectives & Pipeline Workflow

1. **Exploratory Data Analysis**: Inspect historical demand dynamics, rolling trends, and variance.
2. **Classical & STL Decomposition**: Decompose demand into Trend ($T_t$), Seasonality ($S_t$), and Residual ($R_t$) components.
3. **Stationarity Analysis**: Conduct the Augmented Dickey-Fuller (ADF) test, apply differencing, and analyze ACF/PACF correlograms.
4. **Model Training & Hyperparameter Tuning**:
   - Grid search optimal SARIMA/SARIMAX $(p,d,q) \times (P,D,Q)_s$ orders using AIC.
   - Fit Facebook Prophet with weekly seasonality and exogenous event regressors.
5. **Out-of-Sample Evaluation**: Evaluate models on a 30-day holdout split using RMSE, MAE, and MAPE.
6. **Future Projections**: Generate 30-day forward forecast projections with 95% confidence intervals.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, mean_absolute_error
from prophet import Prophet

# Configure plot styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110
print('All libraries imported successfully.')

All libraries imported successfully.


## 2. Load and Inspect Daily Demand Dataset

The dataset tracks historical daily product demand along with exogenous indicators:
- `date`: Observation date
- `demand`: Total units demanded
- `marketing_event`: Binary indicator (1 if marketing campaign active, 0 otherwise)
- `holiday`: Binary indicator (1 if official holiday, 0 otherwise)

In [2]:
data_path = 'data/daily_demand.csv'
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df.set_index('date', inplace=True)

print(f'Total observations: {len(df)}')
print(f'Date range: {df.index.min().strftime("%Y-%m-%d")} to {df.index.max().strftime("%Y-%m-%d")}')
print('\nSummary Statistics:')
print(df.describe().round(2))
print('\nMissing values:\n', df.isnull().sum())

Total observations: 365
Date range: 2025-07-01 to 2026-06-30

Summary Statistics:
       demand  marketing_event  holiday
count  365.00           365.00   365.00
mean    95.45             0.26     0.13
std     28.21             0.44     0.34
min     41.00             0.00     0.00
25%     73.00             0.00     0.00
50%     95.00             0.00     0.00
75%    115.00             1.00     0.00
max    174.00             1.00     1.00

Missing values:
 demand             0
marketing_event    0
holiday            0
dtype: int64
